In [1]:
!pip install pandarallel

  Preparing metadata (setup.py) ... done
  Created wheel for pandarallel: filename=pandarallel-1.6.5-py3-none-any.whl size=16674 sha256=b3fd0cd9c595506532ec31e3c44dc1de0182d604f3323eebb3128407b6fd1f3c
  Stored in directory: /root/.cache/pip/wheels/46/f9/0d/40c9cd74a7cb8dc8fe57e8d6c3c19e2c730449c0d3f2bf66b5
Successfully built pandarallel


In [2]:
import statistics
import pandas as pd
import numpy as np

from sentence_transformers import CrossEncoder, util, SentenceTransformer
from huggingface_hub import login

login()

In [3]:
import pandas as pd

df = pd.read_csv('/content/duplicate_pairs.csv')
display(df.head())

,cluster_label,similarity,item1_id,item2_id,item1_text,item2_text,item1_source,item2_source,item1_book,item2_book,item1_chapter,item2_chapter,item1_verse,item2_verse
0,-1,0.991131,2,31104,"And God said, Let there be light: and there wa...","And God said, “Let there be light,” and there ...",akjv,bsb,Genesis,Genesis,1,1,3,3
1,-1,1.000000,2,62190,"And God said, Let there be light: and there wa...","And God said, Let there be light: and there wa...",akjv,kjv,Genesis,Genesis,1,1,3,3
2,-1,0.855185,7,31109,And God called the firmament Heaven. And the e...,God called the expanse “sky.” And there was ev...,akjv,bsb,Genesis,Genesis,1,1,8,8
3,-1,1.000000,7,62195,And God called the firmament Heaven. And the e...,And God called the firmament Heaven. And the e...,akjv,kjv,Genesis,Genesis,1,1,8,8
4,-1,0.967830,15,31117,And God made two great lights; the greater lig...,God made two great lights: the greater light t...,akjv,bsb,Genesis,Genesis,1,1,16,16


In [4]:


'''
query_response_pairs: list of list that consists of a query and a response to it

return:
similarity score: Cross Encoder Result
'''

similarity_threshhold_en = 0.95
similarity_threshhold_fr = 0.9
eval_model_en = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
eval_model_fr = CrossEncoder('antoinelouis/crossencoder-distilcamembert-mmarcoFR')
# model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def sigmoid(z):
    return 1/(1 + np.exp(-z))

def crossEncoderRes(doc_pair, lang):
  if lang == 'fr':
    eval_model = eval_model_fr
  elif lang == 'en':
    eval_model = eval_model_en


  # embedding_1= model.encode(doc_pair[0], convert_to_tensor=True)
  # embedding_2 = model.encode(doc_pair[1], convert_to_tensor=True)
  # cosine = util.pytorch_cos_sim(embedding_1, embedding_2)

  ### Compute the semantic similarity score using cross encoder
  similarity = eval_model.predict([doc_pair])
  similarity = sigmoid(similarity)

  return similarity #, cosine


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.59k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/811k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/354 [00:00<?, ?B/s]

In [5]:
import tqdm
tqdm.tqdm.pandas()

# Apply crossEncoderRes with progress bar (moved from previous cell 9ca20a55)
df['cross_encoder_similarity'] = df.progress_apply(lambda row: crossEncoderRes([row['item1_text'], row['item2_text']], 'en'), axis=1)

# Extract the similarity value (original content of this cell)
df['cross_encoder_similarity_value'] = df['cross_encoder_similarity'].apply(lambda x: x[0])

# Filter the DataFrame based on the similarity threshold (original content of this cell, using variable)
filtered_df = df[df['cross_encoder_similarity_value'] > similarity_threshhold_en]

display(filtered_df)

100%|██████████| 176271/176271 [25:06<00:00, 117.01it/s]


,cluster_label,similarity,item1_id,item2_id,item1_text,item2_text,item1_source,item2_source,item1_book,item2_book,item1_chapter,item2_chapter,item1_verse,item2_verse,cross_encoder_similarity,cross_encoder_similarity_value
0,-1,0.991131,2,31104,"And God said, Let there be light: and there wa...","And God said, “Let there be light,” and there ...",akjv,bsb,Genesis,Genesis,1,1,3,3,[0.9999746],0.999975
1,-1,1.000000,2,62190,"And God said, Let there be light: and there wa...","And God said, Let there be light: and there wa...",akjv,kjv,Genesis,Genesis,1,1,3,3,[0.99997437],0.999974
2,-1,0.855185,7,31109,And God called the firmament Heaven. And the e...,God called the expanse “sky.” And there was ev...,akjv,bsb,Genesis,Genesis,1,1,8,8,[0.9976488],0.997649
3,-1,1.000000,7,62195,And God called the firmament Heaven. And the e...,And God called the firmament Heaven. And the e...,akjv,kjv,Genesis,Genesis,1,1,8,8,[0.9999572],0.999957
4,-1,0.967830,15,31117,And God made two great lights; the greater lig...,God made two great lights: the greater light t...,akjv,bsb,Genesis,Genesis,1,1,16,16,[0.9999497],0.999950
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176266,6352,0.893802,55699,117893,"Truly I tell you, anyone who does not receive ...","Most certainly I tell you, whoever will not re...",bsb,web,Mark,Mark,10,10,15,15,[0.99987924],0.999879
176267,6352,0.888974,55699,118994,"Truly I tell you, anyone who does not receive ...","Most certainly, I tell you, whoever doesn’t re...",bsb,web,Mark,Luke,10,18,15,17,[0.99986064],0.999861
176268,6352,0.893802,56798,117893,"Truly I tell you, anyone who does not receive ...","Most certainly I tell you, whoever will not re...",bsb,web,Luke,Mark,18,10,17,15,[0.99987924],0.999879
176269,6352,0.888974,56798,118994,"Truly I tell you, anyone who does not receive ...","Most certainly, I tell you, whoever doesn’t re...",bsb,web,Luke,Luke,18,18,17,17,[0.99986064],0.999861


### Narrowing Down the Dataset

Let's filter the `filtered_df` to focus on specific books and sources as requested.

In [15]:
# Define the allowed books and source
allowed_books = ['Matthew', 'Mark', 'Luke', 'John']
allowed_source = 'web'

# Filter the DataFrame based on the conditions
narrowed_filtered_df = filtered_df[
    (filtered_df['item1_book'].isin(allowed_books)) &
    (filtered_df['item2_book'].isin(allowed_books)) &
    (filtered_df['item1_source'] == allowed_source) &
    (filtered_df['item2_source'] == allowed_source)
]

print(f"Original filtered_df size: {len(filtered_df)} rows")
print(f"Narrowed filtered_df size: {len(narrowed_filtered_df)} rows")
display(narrowed_filtered_df.head())

Original filtered_df size: 174592 rows
Narrowed filtered_df size: 537 rows


,cluster_label,similarity,item1_id,item2_id,item1_text,item2_text,item1_source,item2_source,item1_book,item2_book,item1_chapter,item2_chapter,item1_verse,item2_verse,cross_encoder_similarity,cross_encoder_similarity_value
38417,-1,0.856758,116521,117525,They immediately left the boat and their fathe...,"Immediately he called them, and they left thei...",web,web,Matthew,Mark,4,1,22,20,[0.99961936],0.999619
38418,-1,0.861782,116702,118709,"When the demon was cast out, the mute man spok...","He was casting out a demon, and it was mute. I...",web,web,Matthew,Luke,9,11,33,14,[0.9992199],0.999220
38419,-1,1.000000,116755,118508,Blessed is he who finds no occasion for stumbl...,Blessed is he who finds no occasion for stumbl...,web,web,Matthew,Luke,11,7,6,23,[0.9999219],0.999922
38420,-1,0.869269,116880,120192,"Jesus said to them, “Have you understood all t...","Jesus therefore said to them, “Children, have ...",web,web,Matthew,John,13,21,51,5,[0.99883145],0.998831
38421,-1,0.853505,116955,117792,"Jesus summoned his disciples and said, “I have...","“I have compassion on the multitude, because t...",web,web,Matthew,Mark,15,8,32,2,[0.99897873],0.998979


In [16]:
# Initialize the OpenAI client for the LLM layer
from openai import OpenAI
import os
from google.colab import userdata # Uncomment if using Colab userdata

# IMPORTANT: Configure your OpenAI API key securely.
# Recommended: Use Colab's secrets manager (named 'OPENAI_API_KEY')
client = OpenAI(api_key=userdata.get("api_key"))

# Alternatively, use an environment variable (e.g., set via os.environ['OPENAI_API_KEY'])
# client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# For testing, you can directly set it, but avoid this in production:
# client = OpenAI(api_key="YOUR_OPENAI_API_KEY") # <--- Replace with your actual key or secure loading method

### Test OpenAI API

Let's make a simple call to the OpenAI API to confirm the client is initialized correctly and the API key is valid.

In [21]:
try:
    test_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": "Say 'hello'"}
        ]
    )
    print("OpenAI API Test Successful!")
    print(test_response.choices[0].message.content)
except Exception as e:
    print(f"OpenAI API Test Failed: {e}")

OpenAI API Test Successful!
Hello! How can I assist you today?


In [23]:
import json

def identify_document_relationship(relationship_type):
    """
    Identifies the relationship type between two documents.
    """
    return relationship_type

def check_duplications(doc_pair):
    # Step 1: send the conversation and available functions to the model
    content = """ Analyze the relationship between these two documents and identify exactly one of the following cases:
    1. The documents are semantically identical (they convey the exact same meaning).
    2. One document is included in the other (one is a subset or superset of the other's content).
    3. The documents contain conflicting information.
    4. None of the above (no specific relationship from the types listed).

    Due to the nature of type of document, different subject in sentence means they are irrelevant even if they look similiar.

    For cases 1, 2, or 3, please also provide a brief natural language analysis of the identified relationship in your response, in addition to calling the function.

    Document 1:
    {}

    Document 2:
    {}

    You must call a function corresponding to your response.
    """.format(doc_pair[0], doc_pair[1])

    messages = [{"role": "user", "content": content}]
    tools = [
        {
            "type": "function",
            "function": {
                "name": "identify_document_relationship",
                "description": "Identifies the relationship between two documents.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "relationship_type": {
                            "type": "string",
                            "enum": [
                                "semantically_identical",
                                "one_doc_included",
                                "conflict_in_information",
                                "none_of_above"
                            ],
                            "description": "The identified relationship type between the two documents.",
                        },
                    },
                    "required": ["relationship_type"]
                },
            }
        }
    ]

    # Assuming 'client' is an initialized OpenAI client object
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini", # Changed from gpt-5-nano to gpt-4o-mini
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
    except Exception as e:
        return f"Error calling OpenAI API: {e}"

    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    # Step 2: check if the model wanted to call a function
    if tool_calls:
        # Step 3: call the function
        available_functions = {
            "identify_document_relationship": identify_document_relationship,
        }
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_to_call = available_functions[function_name]
            function_args = json.loads(tool_call.function.arguments)

            relationship_type = function_to_call(relationship_type=function_args.get("relationship_type"))
            analysis_text = response_message.content if response_message.content else ""

            return {
                "relationship_type": relationship_type,
                "analysis": analysis_text
            }

    # Fallback if no tool call is made (should not happen with tool_choice="auto" and a well-defined prompt)
    return {"relationship_type": "unknown", "analysis": "No specific relationship identified by the model tool call or analysis provided."}

In [24]:
import tqdm
tqdm.tqdm.pandas()

from pandarallel import pandarallel
pandarallel.initialize(nb_workers=20, progress_bar=True)

# Apply the check_duplications function to each row of the filtered_df
# This will call the LLM for each pair, which can be time-consuming.
def apply_check_duplications(row):
    doc_pair = [row['item1_text'], row['item2_text']]
    result = check_duplications(doc_pair)

    if isinstance(result, str): # Check if result is an error string
        return pd.Series({
            'llm_relationship_type': 'API_ERROR',
            'llm_analysis': result # The error message itself
        })
    else: # result is a dictionary as expected
        return pd.Series({
            'llm_relationship_type': result.get('relationship_type'),
            'llm_analysis': result.get('analysis')
        })

llm_results = narrowed_filtered_df.parallel_apply(apply_check_duplications, axis=1)

# Join the results back to the filtered_df
final_df = pd.concat([narrowed_filtered_df, llm_results], axis=1)
display(final_df.head())

INFO: Pandarallel will run on 20 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


Exception ignored in: <built-in method acquire of _thread.lock object at 0x7902e3903280>
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/popen_fork.py", line 66, in _launch
    self.pid = os.fork()
               ^^^^^^^^^
KeyboardInterrupt: 


,cluster_label,similarity,item1_id,item2_id,item1_text,item2_text,item1_source,item2_source,item1_book,item2_book,item1_chapter,item2_chapter,item1_verse,item2_verse,cross_encoder_similarity,cross_encoder_similarity_value,llm_relationship_type,llm_analysis
38417,-1,0.856758,116521,117525,They immediately left the boat and their fathe...,"Immediately he called them, and they left thei...",web,web,Matthew,Mark,4,1,22,20,[0.99961936],0.999619,none_of_above,
38418,-1,0.861782,116702,118709,"When the demon was cast out, the mute man spok...","He was casting out a demon, and it was mute. I...",web,web,Matthew,Luke,9,11,33,14,[0.9992199],0.999220,semantically_identical,
38419,-1,1.000000,116755,118508,Blessed is he who finds no occasion for stumbl...,Blessed is he who finds no occasion for stumbl...,web,web,Matthew,Luke,11,7,6,23,[0.9999219],0.999922,semantically_identical,
38420,-1,0.869269,116880,120192,"Jesus said to them, “Have you understood all t...","Jesus therefore said to them, “Children, have ...",web,web,Matthew,John,13,21,51,5,[0.99883145],0.998831,none_of_above,
38421,-1,0.853505,116955,117792,"Jesus summoned his disciples and said, “I have...","“I have compassion on the multitude, because t...",web,web,Matthew,Mark,15,8,32,2,[0.99897873],0.998979,conflict_in_information,


In [25]:
# Save the results to a CSV file
final_df.to_csv('llm_analyzed_duplicates.csv', index=False, encoding='utf-8-sig')
print('Processed data with LLM analysis saved to llm_analyzed_duplicates.csv')

# Automatically download the results
from google.colab import files
files.download('llm_analyzed_duplicates.csv')

Processed data with LLM analysis saved to llm_analyzed_duplicates.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>